### Combining the two API Requests

We will use the [AeroDataBox API](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/), which can retrieve all sorts of information about flights and airports. 

For your own github repo, you should clean up the code and keep only relevant functions.

### ### 1. Step. Arrival data from Aerodatabox

### 
_____

We can start by loading our needed secrets and reading the `'cities'` table from Gans's database.

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

In [ ]:
load_dotenv()
connection_string = os.getenv("CON_STRING")
rapidapi_key = os.getenv("RAPIDAPI_KEY")
cities_df = pd.read_sql("cities", con=connection_string)
cities_df

### Step 1: Airports

A function for finding airports is already provided. We'll modify it just a little, selecting only a few columns from each result and renaming one of them.

In [ ]:
def get_airports(cities_df):
    # API headers
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
    url = "https://aerodatabox.p.rapidapi.com/airports/search/location"
    

    # DataFrame to store results
    all_airports = []

    for _, row in cities_df.iterrows():
    # Construct the URL with the latitude and longitude
        querystring = {"lat":row["latitude"],"lon":row["longitude"],"radiusKm":"50","limit":"10","withFlightInfoOnly":"true"}

        # Make the API request
        response = requests.get(url, headers=headers, params=querystring)

        if response.status_code == 200:
            data = response.json()
            airports = pd.json_normalize(data.get('items', []))
            all_airports.append(airports)
        else:
            print(f"WARNING: Failed to retrieve airports for {row['name']}")
    print(response)
    return pd.concat(all_airports, ignore_index=True)

In [ ]:
airports_df, response = get_airports(cities_df)
airports_df

This result looks just fine. Before proceeding with flights, let's write a table definition to `03_gans_schema.sql` and populate the table with `airports_df`. 

In [ ]:
#airports_df.to_sql(
#    "airports",
#   con=connection_string,
#    if_exists="append",
#    index=False
#)

### Step 2: Flights

Let's begin by reading from the database's `'airports'` table. Here, we select only airports that we've flagged as "active".

In [ ]:
airports_df = pd.read_sql("airports" , con=connection_string)
airports_df

#### One airport, one time window

Following the approach taken with weather forecasts, we'll start small with a single call to the API and then add layers of loops to retrieve more and more data.

In [ ]:
# copy headers from above
headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }

# copy url and querystring from AeroDataBox website, filling in one icao code
url = "https://aerodatabox.p.rapidapi.com/flights/airports/icao/EDDB/2026-08-22T00:00/2026-08-22T11:59"
querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}

response = requests.get(url, headers=headers, params=querystring)
print(response.json())

We can now begin to explore the response

In [ ]:
response_json = response.json()
response_json.keys()

In [ ]:
# looks like a list
response_json["arrivals"]

In [ ]:
# try to understand one arrival
# check the webpage as well, there may be items not present in every arrival!
arrival = response_json["arrivals"][0]
arrival

In [ ]:
arrival.keys()

In [ ]:
arrival_dict = {
    "depart_airport": arrival["movement"]["airport"]["name"],
    "depart_country": arrival["movement"]["airport"]["countryCode"].upper(), # country codes are more often all capital letters
    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
    "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
    "flight_number": arrival["number"],
    "aircraft": arrival["aircraft"]["model"]
}
arrival_dict

Now let's see if this works with the rest of the flights in the response.

In [ ]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"]["revisedTime"]["local"],
        "flight_number": arrival["number"],
        "aircraft": arrival["aircraft"]["model"]
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)

Looks like we get the same problem we saw with weather: not every key is included in every arrival. `.get()` will help us here again.

In [ ]:
arrivals = [] # empty list to store arrivals
for arrival in response_json["arrivals"]:
    arrival_dict = {
        "depart_airport": arrival["movement"]["airport"]["name"],
        "depart_country": arrival["movement"]["airport"]["countryCode"].upper(),
        "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
        "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
        "flight_number": arrival["number"],
        "aircraft": arrival["aircraft"].get("model", None)
    }
    arrivals.append(arrival_dict)

flights_df = pd.DataFrame(arrivals)
flights_df

Our API will only let us retrieve flight data for up to 12 hours at a time. To cover all of tomorrow, we'll need to build a `for` loop. We'll also need to work out how to get tomorrow's date.

#### One airport, two time windows

In [ ]:
today = pd.Timestamp.now().date()
one_day = pd.Timedelta(1, "day")
tomorrow = today+one_day
tomorrow_str = tomorrow.strftime("%Y-%m-%d")
tomorrow_str

In [ ]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]
airport = airports_df.loc[0] # use the DataFrame this time, instead of copying call details from website

arrivals = [] # empty list to store arrivals
for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport['icao']}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
    querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
    response = requests.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
        data = response.json()["arrivals"]
        for arrival in data:
            arrival_dict = {
                "arrive_icao": airport["icao"], # add foreign-key information
                "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                "depart_airport": arrival["movement"]["airport"].get("name", None),
                "depart_country": arrival["movement"]["airport"].get("countryCode", "XX").upper(),
                "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                "flight_number": arrival.get("number", None),
                "aircraft": arrival.get("aircraft", {}).get("model", None)
            }
            arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df

#### All airports, two time windows

In [ ]:
headers = { # headers stay constant
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "aerodatabox.p.rapidapi.com",
        "Content-Type": "application/json"
    }
times = [["00:00", "11:59"], ["12:00", "23:59"]]

arrivals = [] # empty list to store arrivals
for _, airport in airports_df.iterrows():
    for start_time, end_time in times: # we can "unpack" the interior lists into two iteration variables
        url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{airport['icao']}/{tomorrow_str}T{start_time}/{tomorrow_str}T{end_time}"
        querystring = {"withLeg":"false","direction":"Arrival","withCancelled":"false","withCodeshared":"false"}
        response = requests.get(url, headers=headers, params=querystring)
    
        if response.status_code == 200:
            data = response.json()["arrivals"]
            for arrival in data:
                arrival_dict = {
                    "arrive_icao": airport["icao"], # add foreign-key information
                    "depart_icao": arrival["movement"]["airport"].get("icao", None), # add this since countryCode and maybe name aren't reliable
                    "depart_airport": arrival["movement"]["airport"].get("name", None),
                    "depart_country": arrival["movement"]["airport"].get("countryCode", None), # move .upper() to deal with NoneTypes
                    "arrive_time_scheduled": arrival["movement"]["scheduledTime"]["local"],
                    "arrive_time_revised": arrival["movement"].get("revisedTime", {}).get("local", None), 
                    "flight_number": arrival.get("number", None),
                    "aircraft": arrival.get("aircraft", {}).get("model", None)
                }
                arrivals.append(arrival_dict)
        
flights_df = pd.DataFrame(arrivals)
flights_df["arrive_time_scheduled"] = pd.to_datetime(flights_df["arrive_time_scheduled"].str[:-6])
flights_df["arrive_time_revised"] = pd.to_datetime(flights_df["arrive_time_revised"].str[:-6])
flights_df["depart_country"] = flights_df["depart_country"].str.upper()
flights_df

Success! Now add a table definition to the database before pushing the data.

In [ ]:
# we get "Incorrect datetime value: '2026-08-22 05:40+02:00' for column 'arrive_time_scheduled' at row 1", so let's convert to datetime
flights_df.info()

In [ ]:
#flights_df.to_sql(
#    "flights",
#    con=connection_string,
#    if_exists="append",
#    index=False
#)